# 分子对接图搜索实验

本示例通过九章 SDK 的本地应用接口加载分子对接图，使用后选择样本生成候选子图，并通过 clique shrink/search 得到团结构候选。

## 1. 初始化环境

In [ ]:
import numpy as np

from jiuzhang.local.applications import (
    clique_search,
    clique_shrink,
    draw_graph,
    is_clique,
    load_phat_graph,
    load_tace_as_graph,
    postselect_subgraphs,
    subgraph_density_summary,
)

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Sarasa UI SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 2. 加载 TACE-AS 图数据

In [ ]:
tace_as = load_tace_as_graph()
print(tace_as.name, tace_as.adjacency.shape)

draw_graph(tace_as.adjacency, title='TACE-AS graph')
plt.show()

## 3. 后选择采样并比较图密度

`min_photons` 和 `max_photons` 控制保留样本的总光子数范围，返回的每个样本会转成一个候选节点集合。

In [ ]:
candidate_subgraphs = postselect_subgraphs(tace_as, min_photons=8, max_photons=8)
summary = subgraph_density_summary(tace_as.adjacency, candidate_subgraphs, seed=7)

print('Candidate count:', len(candidate_subgraphs))
print('Sampled mean density: {:.4f}'.format(summary['sampled_mean_density']))
print('Uniform mean density: {:.4f}'.format(summary['uniform_mean_density']))

## 4. 团结构收缩与局部搜索

In [ ]:
shrunk = [clique_shrink(tace_as.adjacency, nodes) for nodes in candidate_subgraphs[:50]]
searched = [clique_search(tace_as.adjacency, nodes, iterations=10) for nodes in shrunk]
clique_sizes = [len(nodes) for nodes in searched]
first_clique = searched[int(np.argmax(clique_sizes))]

print('First candidate is clique:', is_clique(tace_as.adjacency, first_clique))
print('First ten clique sizes:', clique_sizes[:10])
print('Best clique size:', max(clique_sizes))
print('Best clique nodes:', first_clique)

draw_graph(tace_as.adjacency, first_clique, title='TACE-AS clique candidate')
plt.show()

## 5. PHat 图上的最大团搜索示例

In [ ]:
phat = load_phat_graph()
phat_subgraphs = postselect_subgraphs(phat, min_photons=16, max_photons=20)
phat_shrunk = [clique_shrink(phat.adjacency, nodes) for nodes in phat_subgraphs[:100]]
phat_searched = [clique_search(phat.adjacency, nodes, iterations=10) for nodes in phat_shrunk]
phat_sizes = [len(nodes) for nodes in phat_searched]
largest_clique = phat_searched[int(np.argmax(phat_sizes))]

print('Largest clique found:', largest_clique)
print('Largest clique size:', len(largest_clique))

draw_graph(phat.adjacency, largest_clique, title='PHat clique candidate')
plt.show()